In [1]:
import requests
import pandas as pd
import folium
import numpy as np

# 1. 서울시 열린데이터광장 인증키 및 API 설정
seoul_key = "65464c617a63686f37366f67686370"
service_name = "TvEmgcHospitalInfo"
url = f"http://openAPI.seoul.go.kr:8088/{seoul_key}/json/{service_name}/1/100/"

print("🚀 서울시 오픈 API 서버에 실시간 데이터를 요청 중...")

hospital_list = []
try:
    response = requests.get(url)
    print("HTTP 상태 코드:", response.status_code)
    
    if response.status_code == 200:
        data = response.json()
        
        # API 정상 응답 구조 확인 및 데이터 추출
        if service_name in data and 'row' in data[service_name]:
            row_list = data[service_name]['row']
            print(f"✅ 성공! 수신된 원본 응급실 데이터 개수: {len(row_list)}개")
            
            for item in row_list:
                hospital_list.append({
                    '기관명': item.get('DUTYNAME'),
                    '주소': item.get('DUTYADDR'),
                    '병원위도': pd.to_numeric(item.get('WGS84LAT'), errors='coerce'),
                    '병원경도': pd.to_numeric(item.get('WGS84LON'), errors='coerce')
                })
        else:
            print("⚠️ API 응답 데이터 형태 확인 필요, 안전 백업 모드로 전환합니다.")
    else:
        print("❌ API 요청 실패, 안전 백업 모드로 전환합니다.")
except Exception as e:
    print("❌ API 연동 중 예외 발생:", e)

# 2. 데이터프레임 변환 및 4개 구 필터링 (강남구, 마포구, 영등포구, 종로구)
df_clean = pd.DataFrame(hospital_list)

if not df_clean.empty:
    target_districts = ['강남구', '마포구', '영등포구', '종로구']
    df_filtered = df_clean[df_clean['주소'].str.contains('|'.join(target_districts), na=False)].copy()
else:
    # 만약 API 호출에 문제가 생길 경우를 대비한 안전 백업 데이터프레임
    df_filtered = pd.DataFrame(columns=['기관명', '주소', '병원위도', '병원경도'])

print(f"🎯 4개 구(강남, 마포, 영등포, 종로) 필터링 완료: 총 {len(df_filtered)}개 병원 확인")


# 3. 실시간 혼잡도/잔여 상태 시뮬레이션 부여 (추후 실제 잔여 병상 컬럼으로 교체 가능)
np.random.seed(42)
if not df_filtered.empty:
    # 70% 확률로 여유(정상), 30% 확률로 혼잡(만실) 부여
    df_filtered['is_crowded'] = np.random.choice([True, False], size=len(df_filtered), p=[0.3, 0.7])
else:
    pass


# 4. Folium 지도 시각화 (깔끔한 HTML 팝업 + 3km 반경 원)
m = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

for idx, row in df_filtered.iterrows():
    is_crowded = row['is_crowded']
    
    if is_crowded:
        marker_color = 'red'
        status_msg = "🔴 혼잡 / 만실"
    else:
        marker_color = 'green'
        status_msg = "🟢 여유 / 정상"
        
    # 팝업 디자인 박스
    popup_html = f"""
    <div style="width: 240px; font-family: 'Malgun Gothic', sans-serif; padding: 5px;">
        <h4 style="margin: 0 0 8px 0; color: #222; font-size: 14px;"><b>{row['기관명']}</b></h4>
        <p style="margin: 3px 0; font-size: 12px; color: #555;"><b>주소:</b> {row['주소']}</p>
        <hr style="border: 0; border-top: 1px solid #ddd; margin: 6px 0;">
        <p style="margin: 3px 0; font-size: 13px;"><b>상태:</b> {status_msg}</p>
    </div>
    """
    
    # 병원 마커 추가
    folium.Marker(
        location=[row['병원위도'], row['병원경도']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row['기관명'],
        icon=folium.Icon(color=marker_color, icon='hospital-o', prefix='fa')
    ).add_to(m)
    
    # 3km 커버리지 원 추가
    folium.Circle(
        location=[row['병원위도'], row['병원경도']],
        radius=3000,
        color='blue',
        weight=1,
        fill=True,
        fill_color='blue',
        fill_opacity=0.05
    ).add_to(m)

# 최종 지도 출력
m

🚀 서울시 오픈 API 서버에 실시간 데이터를 요청 중...
HTTP 상태 코드: 200
✅ 성공! 수신된 원본 응급실 데이터 개수: 77개
🎯 4개 구(강남, 마포, 영등포, 종로) 필터링 완료: 총 21개 병원 확인


In [2]:
# 지도를 만든 셀 바로 다음 셀에서 실행하세요
m.save("index.html")
print("✅ index.html 저장 완료!")

✅ index.html 저장 완료!
